# Long-term treatment (compositional PK/PD)

**What you will learn:** compute a **viability kernel** with `rci`, use it as a
reach target, and switch controllers when the therapeutic level is crossed.

**pyspect API:** `TVHJImpl`, `rci`, `reach`

**Prerequisites:** [`pkpd.ipynb`](pkpd.ipynb)

Same patient model as `pkpd.ipynb`, but the treatment must **reach and then hold**
the therapeutic level indefinitely.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp

from pyspect.impls.hj_reachability import TVHJImpl
from pyspect.systems.hj_reachability import PKPD

In [ ]:
# Same setup as hjr_examples: 7-day horizon, 4 decision points per day
POINTS_PER_DAY = 4
T = 7
N = POINTS_PER_DAY * T

AXES = [
    dict(name='t',  bounds=[0, T], points=N + 1),
    dict(name='x1', bounds=[0, 1], points=26),
    dict(name='x2', bounds=[0, 1], points=26),
    dict(name='x3', bounds=[0, 5], points=26),
]

# This variant clears the effect slowly (gamma) and the tissue faster (delta)
impl = TVHJImpl(dict(cls=PKPD, gamma=0.05, delta=0.5), AXES, accuracy='very_high')

S = impl.grid.states
X1, X2, X3 = S[..., 0], S[..., 1], S[..., 2]

# Good states: therapeutic AND non-toxic in blood AND non-toxic in tissue
g0 = jnp.maximum(jnp.maximum(0.5 - X1, X2 - 0.8), X3 - 0.8)

# Safe states only, used as the constraint of the second stage
g = jnp.maximum(X2 - 0.8, X3 - 0.8)

## Stage 1 - where can the patient be held?

The original script runs the solver with a `max(v, g0)` post-processor and no target,
which is the definition of a viability kernel. `TVHJImpl` exposes it directly as `rci`,
since staying inside a set forever is the same as never being forced out of it.


In [ ]:
V0 = impl.rci(g0)
V0_np = np.array(V0)

print('V0', V0_np.shape)
print(f'holdable at full horizon: {100 * (V0_np[0] <= 0).mean():.2f} % of the grid')
print(f'good states to begin with: {100 * float((g0 <= 0).mean()):.2f} %')

In [ ]:
# Cross-check against the recursion written out by hand in the original script
import hj_reachability as hj

settings = hj.SolverSettings.with_accuracy(
    'very_high', value_postprocessor=lambda t, v: jnp.maximum(v, g0))
V0_ref = np.array(hj.solve(settings, PKPD(gamma=0.05, delta=0.5), impl.grid,
                           np.linspace(0.0, -T, N + 1), g0))

print('largest disagreement:', np.abs(V0_np - V0_ref[::-1]).max())

## Stage 2 - how to get there

The viability kernel becomes the target of a reach-avoid, with the toxic thresholds as
the constraint. Note that the target is a level set produced by a previous solve, not a
hand-written formula: this is the whole point of composing.


In [ ]:
l = V0[0]                # the kernel, at full budget
V = impl.reach(l, g)
V_np = np.array(V)

ttg = np.array(impl.timeline)[::-1]
print('V', V_np.shape)
print(f'treatable at full horizon: {100 * (V_np[0] <= 0).mean():.2f} % of the grid')
print(f'value at the origin (untreated patient): {V_np[0, 0, 0, 0]:+.3f}')

In [ ]:
x1v = np.array(impl.grid.coordinate_vectors[0])
x2v = np.array(impl.grid.coordinate_vectors[1])
KW = dict(origin='lower', aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
EXT = [x1v[0], x1v[-1], x2v[0], x2v[-1]]

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for ax, W, name in [(axs[0], V0_np, 'Stage 1: can be held forever'),
                    (axs[1], V_np,  'Stage 2: can get there and then be held')]:
    sl = W[0, :, :, 0]
    im_ = ax.imshow(sl.T, extent=EXT, **KW)
    ax.contour(x1v, x2v, sl.T, levels=[0], colors='black', linewidths=1.8)
    ax.axvline(0.5, color='green', ls='--', lw=1.5, label='Therapeutic level')
    ax.axhline(0.8, color='red', ls='--', lw=1.5, label='Toxic threshold')
    ax.plot(0, 0, 'ko', ms=9, label='Patient at day 0')
    plt.colorbar(im_, ax=ax, label=r'$V$ (clipped)')
    ax.set_xlabel(r'$x_1$  (effect)'); ax.set_ylabel(r'$x_2$  (blood)')
    ax.set_title(name)
    ax.legend(loc='lower left', framealpha=1.0, fontsize=9)

fig.suptitle(r'Slice $x_3 = 0$')
plt.tight_layout(); plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(5.2, 4.6))

def update(i):
    ax.clear()
    sl = V_np[i, :, :, 0]
    ax.imshow(sl.T, extent=EXT, **KW)
    ax.contour(x1v, x2v, sl.T, levels=[0], colors='black', linewidth=1.4)
    ax.contour(x1v, x2v, V0_np[0, :, :, 0].T, levels=[0],
               colors='green', linewidths=1.2, linestyles='--')
    ax.axhline(0.8, color='red', ls='--', lw=1.0)
    ax.plot(0, 0, 'ko', ms=5)
    ax.set_title(f'Budget = {ttg[i]:.1f} days')
    ax.set_xlabel(r'$x_1$  (effect)')
    ax.set_ylabel(r'$x_2$  (blood)')

ani = FuncAnimation(fig, update, frames=len(V_np), interval=150, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())


## Closed-loop treatment

Two phases, mirroring the two solves. While the patient is below the therapeutic level
the dose follows the gradient of `V`; once the level is crossed the controller switches
to the gradient of `V0`, whose job is only to hold position. The horizon of the second
phase is unbounded in principle, so its value function does not depend on time.


In [ ]:
dyn = impl.reach_dynamics
grads_V = [impl.grid.grad_values(jnp.asarray(V_np[i])) for i in range(N)]
grad_V0 = impl.grid.grad_values(jnp.asarray(V0_np[0]))

def rhs(u):
    def f(t, x):
        xs = jnp.asarray(x)
        return np.array(dyn.open_loop_dynamics(xs, 0.0)
                        + dyn.control_jacobian(xs, 0.0) @ jnp.array([u]))
    return f

def dose(gr_field, x):
    gr = impl.grid.interpolate(gr_field, state=jnp.asarray(x))
    return float(dyn.optimal_control(jnp.asarray(x), 0.0, gr)[0])

ts, ys, us = [], [], []
x0 = np.array([0.0, 0.0, 0.0])
switch = None

for i in range(2 * N):
    if switch is None:
        u = dose(grads_V[min(i, N - 1)], x0)
    else:
        u = dose(grad_V0, x0)

    sol = solve_ivp(rhs(u), [i / POINTS_PER_DAY, (i + 1) / POINTS_PER_DAY],
                    x0, max_step=0.01)
    x0 = sol.y[:, -1]
    ts.append(sol.t if i == 0 else sol.t[1:])
    ys.append(sol.y if i == 0 else sol.y[:, 1:])
    us.append(u)

    if switch is None and x0[0] > 0.5:
        switch = (i + 1) / POINTS_PER_DAY

t_sol = np.concatenate(ts)
y_sol = np.concatenate(ys, axis=1)
u_sol = np.array(us)

print(f'therapeutic level crossed on day {switch:.2f}')
print(f'peak blood  : {y_sol[1].max():.3f}  (toxic above 0.8)')
print(f'peak tissue : {y_sol[2].max():.3f}  (toxic above 0.8)')
print(f'effect at day {2 * T}: {y_sol[0, -1]:.3f}  (must stay above 0.5)')
print(f'lowest effect after the switch: '
      f'{y_sol[0][t_sol >= switch].min():.3f}')

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(7, 6), sharex=True)

axs[0].plot(np.arange(2 * N) / POINTS_PER_DAY, u_sol, color='black',
            drawstyle='steps-post')
axs[0].set_ylabel('Dose\n(normalized)')
axs[0].set_ylim([-0.05, 1.05])

axs[1].plot(t_sol, y_sol[1], color='magenta', label='Blood')
axs[1].plot(t_sol, y_sol[2], color='purple', label='Tissue')
axs[1].axhline(0.8, color='red', ls='--', label='Toxic')
axs[1].set_ylabel('[Drug]\n(normalized)')
axs[1].set_ylim([-0.05, 1.55])
axs[1].legend(loc='center left', bbox_to_anchor=(1, 0.5))

axs[2].plot(t_sol, y_sol[0], color='blue')
axs[2].axhline(0.5, color='green', ls='--', label='Therapeutic')
axs[2].set_ylabel('[X]\n(normalized)')
axs[2].set_ylim([-0.05, 1.05])
axs[2].set_xlabel('Day')
axs[2].set_xlim(0, 2 * T)
axs[2].legend(loc='center left', bbox_to_anchor=(1, 0.5))

for ax in axs:
    ax.axvline(switch, color='gray', lw=1.2, ls=':')

fig.suptitle('Reach the therapeutic level, then hold it')
plt.tight_layout(); plt.show()